# V6.2 — V6.03-style \(E(a,\alpha)\) and energy-component figures

This notebook is a new branch. It does **not** overwrite V6.03.

The geometric ensemble is defined by the experimental reference state

\[
L=16\ {\rm nm},\qquad a_{\rm ref}=20.31\ {\rm nm},\qquad
\alpha_{\rm ref}=74.0^\circ .
\]

The reference particle azimuth is the legacy V6.02 orientation, written here as
\(\gamma_{\rm ref}=-15^\circ\) relative to the simple-cubic-aligned frame.
The code evaluates the exact Euclidean nearest distance of two co-oriented
Minkowski rounded cubes at this state and calls it \(D_{\rm ref}\). It does not
insert the old empirical value 1.8 nm.

For every \((\alpha,\gamma)\), the lattice parameter is then obtained from

\[
D_{\min}^{\rm exact}(a,\alpha,\gamma)=D_{\rm ref}.
\]

The twist is periodic over 120° because the field is parallel to the cube
body diagonal `[111]`. The scan reports both the zero-temperature minimum over
\(\gamma\) and the finite-temperature orientational free energy.

In the final figures, \(\gamma\) is an internal relaxation coordinate only.
Each particle size keeps the two detailed V6.03-style multi-panel outputs:
the first combines \(E(a,\alpha)\), \(E(\alpha)\), and \(E(a)\); the second
compares the component curves and the constant-gap geometry. No
\(E(\alpha,\gamma)\) figure is exported.

Model choices retained from V6.03:

- \(N_{\rm voxel}=4\) for the sharp-cube VdW quadrature.
- 3×3×3 / 26-neighbour direct sum for the long-range dipole interaction.
- Six fixed primitive bonds for the short-range VdW term used in the main
  Hamiltonian; these become the six face-aligned bonds at simple cubic.
  The full 26-neighbour VdW sum and its 20-neighbour tail are retained as
  diagnostics.
- The same \(M_s\), Hamaker constant, field and cubic anisotropy.

No fitted ligand-repulsion term is added in this version. The exact reference
gap is imposed as a geometric constraint, so steric energy is identically zero
on the scanned path.

In [ ]:
import time
from pathlib import Path

import numpy as np
from scipy.constants import Boltzmann, mu_0
from scipy.optimize import brentq, minimize_scalar
import matplotlib as mpl
import matplotlib.pyplot as plt


# ============================================================
#  FIGURE STYLE
# ============================================================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "figure.titlesize": 8,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "lines.linewidth": 1.4,
})


# ============================================================
#  USER PARAMETERS — V6.2
# ============================================================
PHYS = {
    "temperature_K": 298.15,
    "Ms": 285000.0,          # A/m
    "B_field": 0.5,          # T
    "K_ani": 2.0e4,          # J/m^3
    "Hamaker": 2.0e-20,      # J
    "N_voxel": 4,
    "roundness_nm": 1.5,
    "vdw_d2_floor_m2": 1.0e-19,
    # Main Hamiltonian: six fixed primitive bonds, face-aligned at SC.
    # The full 26-neighbour VdW sum remains a diagnostic.
    "vdw_neighbour_mode": "primitive6",
}

PARTICLE_SIZES_NM = [11.0, 13.0, 16.0, 18.0, 18.2, 19.5, 29.6]
EXP_DATA = {
    16.0: {"d": 20.31, "alpha": 74.0},
    18.0: {"d": 22.2, "alpha": 73.0},
    18.2: {"d": 23.5, "alpha": 72.0},
    19.5: {"d": 29.0, "alpha": 60.0},
    29.6: {"d": 39.6, "alpha": 60.0},
}

# Q(gamma) = R_x(gamma) @ Q_SC.
# The original V6.02 align_vectors frame is gamma ~= -15 deg.
REFERENCE = {
    "L_nm": 16.0,
    "a_nm": EXP_DATA[16.0]["d"],
    "alpha_deg": EXP_DATA[16.0]["alpha"],
    "gamma_deg": -15.0,
}

SCAN = {
    "alpha_min_deg": 60.0,
    "alpha_max_deg": 90.0,
    "N_alpha": 61,           # 0.5 degree spacing
    "N_gamma": 61,           # ~1.97 degree spacing over one 120 degree period
    "refine_gamma_min": True,
    # N=1 is a local-rotor entropy estimate. For a coherent N-particle
    # domain, increase this value; N -> infinity approaches min_gamma E.
    "N_coherent_particles": 1,
}

OUTPUT_DIR = Path("results") / "V620_EaAlpha"
DIAGNOSTIC_DIR = OUTPUT_DIR / "internal_diagnostics"
FIGURE_DIR = OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print("V6.2 parameters loaded.")

In [ ]:
class ReferenceGapRhombohedralSL:
    '''Rhombohedral lattice of co-oriented rounded nanocubes.'''

    def __init__(self, L_nm, phys=PHYS):
        self.L_nm = float(L_nm)
        self.L = self.L_nm * 1e-9
        self.r_round_nm = float(phys["roundness_nm"])
        self.r_round = min(
            self.r_round_nm * 1e-9,
            self.L / 2.0 - 1e-15,
        )
        self.core_difference_halfwidth = self.L - 2.0 * self.r_round

        self.temperature = float(phys["temperature_K"])
        self.kT = Boltzmann * self.temperature
        self.volume = self.L**3
        self.Ms = float(phys["Ms"])
        self.mu_magnitude = self.Ms * self.volume
        self.mu_hat = np.array([1.0, 0.0, 0.0])
        self.mu_vec = self.mu_magnitude * self.mu_hat
        self.B_field = float(phys["B_field"])
        self.K_ani = float(phys["K_ani"])
        self.Cmag = mu_0 / (4.0 * np.pi)
        self.Aham = float(phys["Hamaker"])
        self.d2_floor = float(phys["vdw_d2_floor_m2"])

        # 26 neighbours in a 3x3x3 cell, excluding the origin.
        grid = np.array(
            np.meshgrid([-1, 0, 1], [-1, 0, 1], [-1, 0, 1], indexing="ij")
        ).T.reshape(-1, 3)
        self.neighbour_indices = grid[np.any(grid != 0, axis=1)]
        self.primitive_face_mask = (
            np.sum(np.abs(self.neighbour_indices), axis=1) == 1
        )
        if np.count_nonzero(self.primitive_face_mask) != 6:
            raise RuntimeError("The primitive VdW shell must contain 6 bonds.")
        primitive_set = {
            tuple(index)
            for index in self.neighbour_indices[self.primitive_face_mask]
        }
        expected_primitive_set = {
            (1, 0, 0), (-1, 0, 0),
            (0, 1, 0), (0, -1, 0),
            (0, 0, 1), (0, 0, -1),
        }
        if primitive_set != expected_primitive_set:
            raise RuntimeError(
                "The primitive VdW mask must be the fixed +/- lattice axes."
            )
        self.vdw_neighbour_mode = str(
            phys.get("vdw_neighbour_mode", "primitive6")
        )
        if self.vdw_neighbour_mode != "primitive6":
            raise ValueError(
                "V6.20 main Hamiltonian requires "
                "vdw_neighbour_mode='primitive6'."
            )

        # Sharp-cube voxel quadrature retained from V6.03.
        self.N_voxel = int(phys["N_voxel"])
        step = self.L / self.N_voxel
        lin = np.linspace(
            -self.L / 2.0 + step / 2.0,
            self.L / 2.0 - step / 2.0,
            self.N_voxel,
        )
        gx, gy, gz = np.meshgrid(lin, lin, lin, indexing="ij")
        self.vox_ref = np.column_stack([gx.ravel(), gy.ravel(), gz.ravel()])
        self.vox_delta = (
            self.vox_ref[:, None, :] - self.vox_ref[None, :, :]
        )
        self.vox_volume = step**3

        # At alpha=90 degrees the lattice basis rows are cube face normals.
        self.Q_SC = self.basis_unit(90.0).T
        if not np.allclose(self.Q_SC.T @ self.Q_SC, np.eye(3), atol=1e-12):
            raise RuntimeError("Q_SC is not orthogonal.")
        if np.linalg.det(self.Q_SC) < 0:
            raise RuntimeError("Q_SC must be a proper rotation.")

    @staticmethod
    def wrap_gamma(gamma_deg):
        '''Map gamma to the periodic interval [-60, 60).'''
        return (float(gamma_deg) + 60.0) % 120.0 - 60.0

    @staticmethod
    def rotation_about_field(gamma_deg):
        g = np.deg2rad(gamma_deg)
        cg, sg = np.cos(g), np.sin(g)
        return np.array([
            [1.0, 0.0, 0.0],
            [0.0, cg, -sg],
            [0.0, sg,  cg],
        ])

    def orientation(self, gamma_deg):
        '''Body-to-lab rotation. gamma=0 is simple-cubic face aligned.'''
        return self.rotation_about_field(self.wrap_gamma(gamma_deg)) @ self.Q_SC

    @staticmethod
    def basis_unit(alpha_deg):
        alpha = np.deg2rad(alpha_deg)
        cp2 = (2.0 * np.cos(alpha) + 1.0) / 3.0
        cp = np.sqrt(max(cp2, 0.0))
        sp = np.sqrt(max(1.0 - cp**2, 0.0))
        return np.array([
            [cp,  sp,       0.0],
            [cp, -0.5*sp,   (np.sqrt(3.0)/2.0)*sp],
            [cp, -0.5*sp,  -(np.sqrt(3.0)/2.0)*sp],
        ])

    def basis(self, a_nm, alpha_deg):
        return self.basis_unit(alpha_deg) * (float(a_nm) * 1e-9)

    def neighbour_vectors(self, a_nm, alpha_deg):
        return self.neighbour_indices @ self.basis(a_nm, alpha_deg)

    def pair_exact_gap(self, r_ij, gamma_deg):
        '''Exact signed Euclidean gap of two co-oriented rounded cubes.'''
        Q = self.orientation(gamma_deg)
        p = np.asarray(r_ij) @ Q
        z = np.abs(p) - self.core_difference_halfwidth
        outside = np.linalg.norm(np.maximum(z, 0.0))
        inside = min(float(np.max(z)), 0.0)
        return outside + inside - 2.0 * self.r_round

    def neighbour_exact_gaps(self, a_nm, alpha_deg, gamma_deg):
        Q = self.orientation(gamma_deg)
        rv = self.neighbour_vectors(a_nm, alpha_deg)
        p = rv @ Q
        z = np.abs(p) - self.core_difference_halfwidth
        outside = np.linalg.norm(np.maximum(z, 0.0), axis=1)
        inside = np.minimum(np.max(z, axis=1), 0.0)
        return outside + inside - 2.0 * self.r_round

    def nearest_exact_gap(self, a_nm, alpha_deg, gamma_deg):
        return float(
            np.min(self.neighbour_exact_gaps(a_nm, alpha_deg, gamma_deg))
            * 1e9
        )

    def neighbour_sharp_gaps(self, a_nm, alpha_deg, gamma_deg):
        '''Exact signed gap of the sharp cubes used by the VdW quadrature.'''
        Q = self.orientation(gamma_deg)
        rv = self.neighbour_vectors(a_nm, alpha_deg)
        p = rv @ Q
        z = np.abs(p) - self.L
        outside = np.linalg.norm(np.maximum(z, 0.0), axis=1)
        inside = np.minimum(np.max(z, axis=1), 0.0)
        return outside + inside

    def nearest_sharp_gap(self, a_nm, alpha_deg, gamma_deg):
        return float(
            np.min(self.neighbour_sharp_gaps(a_nm, alpha_deg, gamma_deg))
            * 1e9
        )

    def nearest_support_gap(self, a_nm, alpha_deg, gamma_deg):
        '''Legacy radial support-plane gap, retained only as a diagnostic.'''
        Q = self.orientation(gamma_deg)
        rv = self.neighbour_vectors(a_nm, alpha_deg)
        rmag = np.linalg.norm(rv, axis=1)
        body_hat = (rv @ Q) / rmag[:, None]
        excluded = (
            self.core_difference_halfwidth
            * np.sum(np.abs(body_hat), axis=1)
            + 2.0 * self.r_round
        )
        return float(np.min(rmag - excluded) * 1e9)

    def a_for_exact_gap(self, alpha_deg, gamma_deg, gap_nm):
        '''Solve D_min^exact(a, alpha, gamma) = gap_nm.'''
        gap_nm = float(gap_nm)
        objective = (
            lambda a_nm:
            self.nearest_exact_gap(a_nm, alpha_deg, gamma_deg) - gap_nm
        )
        lower = max(0.25 * self.L_nm, 0.5)
        upper = max(3.0 * self.L_nm, self.L_nm + gap_nm + 10.0)
        f_lower = objective(lower)
        f_upper = objective(upper)
        while f_lower > 0.0 and lower > 1e-4:
            lower *= 0.5
            f_lower = objective(lower)
        while f_upper < 0.0 and upper < 20.0 * self.L_nm:
            upper *= 1.5
            f_upper = objective(upper)
        if f_lower > 0.0 or f_upper < 0.0:
            raise RuntimeError(
                f"Could not bracket a at alpha={alpha_deg}, gamma={gamma_deg}."
            )
        return float(
            brentq(objective, lower, upper, xtol=1e-9, rtol=1e-12)
        )

    def energy_components(self, a_nm, alpha_deg, gamma_deg):
        '''Energy components in kBT per particle.'''
        Q = self.orientation(gamma_deg)
        rv = self.neighbour_vectors(a_nm, alpha_deg)
        dv = np.linalg.norm(rv, axis=1)

        Ez = -self.mu_magnitude * self.B_field
        mu_body = Q.T @ self.mu_hat
        mx, my, mz = mu_body
        Ea = -self.K_ani * self.volume * (
            (mx*my)**2 + (mx*mz)**2 + (my*mz)**2
        )

        mu2 = self.mu_magnitude**2
        Edd = 0.5 * np.sum(
            self.Cmag
            * (
                mu2 / dv**3
                - 3.0 * (rv @ self.mu_vec)**2 / dv**5
            )
        )

        # Evaluate the original sharp-cube voxel integral in the body frame.
        # VdW is short-ranged in the intended face-contact Hamiltonian:
        # only the six primitive bonds enter the main energy.  The all-26
        # result and its tail-20 remainder are retained as diagnostics.
        rv_body = rv @ Q
        separation = (
            self.vox_delta[None, :, :, :]
            - rv_body[:, None, None, :]
        )
        d2 = np.einsum("...i,...i->...", separation, separation)
        scv = -(self.Aham / np.pi**2) * self.vox_volume**2
        Evdw_per_neighbour = 0.5 * scv * np.sum(
            1.0 / np.maximum(d2, self.d2_floor)**3,
            axis=(1, 2),
        )
        Evdw_face6 = np.sum(
            Evdw_per_neighbour[self.primitive_face_mask]
        )
        Evdw_full26 = np.sum(Evdw_per_neighbour)
        Evdw_tail20 = Evdw_full26 - Evdw_face6

        result = {
            "Zeeman": Ez / self.kT,
            "Anisotropy": Ea / self.kT,
            "Dipole": Edd / self.kT,
            # VdW is the six-face-bond term used by the Hamiltonian.
            "VdW": Evdw_face6 / self.kT,
            "VdW_face6": Evdw_face6 / self.kT,
            "VdW_full26": Evdw_full26 / self.kT,
            "VdW_tail20": Evdw_tail20 / self.kT,
            "Steric": 0.0,
        }
        result["Interaction"] = result["Dipole"] + result["VdW"]
        result["Total_with_single_particle_offsets"] = sum(
            result[key]
            for key in ["Zeeman", "Anisotropy", "Dipole", "VdW", "Steric"]
        )
        # Public Total includes every Hamiltonian contribution once per
        # particle. Interaction remains available as an internal diagnostic.
        result["Total"] = result["Total_with_single_particle_offsets"]
        return result

    def state_at_gap(self, alpha_deg, gamma_deg, gap_nm):
        a_nm = self.a_for_exact_gap(alpha_deg, gamma_deg, gap_nm)
        energy = self.energy_components(a_nm, alpha_deg, gamma_deg)
        return a_nm, energy

In [ ]:
# ============================================================
#  REFERENCE-GAP CALIBRATION AND GEOMETRY UNIT TESTS
# ============================================================
model = ReferenceGapRhombohedralSL(REFERENCE["L_nm"])

D_REF_NM = model.nearest_exact_gap(
    REFERENCE["a_nm"],
    REFERENCE["alpha_deg"],
    REFERENCE["gamma_deg"],
)
# The calculation retains the exact calibrated value.  Figure text follows
# the requested one-decimal display convention.
D_DISPLAY_NM = round(D_REF_NM, 1)
D_REF_SUPPORT_NM = model.nearest_support_gap(
    REFERENCE["a_nm"],
    REFERENCE["alpha_deg"],
    REFERENCE["gamma_deg"],
)

if D_REF_NM <= 0.0:
    raise RuntimeError("The chosen experimental reference state overlaps.")

reference_energy = model.energy_components(
    REFERENCE["a_nm"],
    REFERENCE["alpha_deg"],
    REFERENCE["gamma_deg"],
)
reference_sharp_gap = model.nearest_sharp_gap(
    REFERENCE["a_nm"],
    REFERENCE["alpha_deg"],
    REFERENCE["gamma_deg"],
)

# The reference state must be recovered by the inverse gap calculation.
a_reference_check = model.a_for_exact_gap(
    REFERENCE["alpha_deg"],
    REFERENCE["gamma_deg"],
    D_REF_NM,
)
assert abs(a_reference_check - REFERENCE["a_nm"]) < 1e-6
assert reference_sharp_gap > 0.0
assert abs(
    model.nearest_exact_gap(
        REFERENCE["a_nm"],
        REFERENCE["alpha_deg"],
        REFERENCE["gamma_deg"] + 120.0,
    )
    - D_REF_NM
) < 1e-10

# At alpha=90, gamma=0 must be a true face-to-face simple cubic state.
B90 = model.basis_unit(90.0)
Q0 = model.orientation(0.0)
face_alignment = np.abs(B90 @ Q0)
assert np.allclose(face_alignment, np.eye(3), atol=1e-12)
a_sc_check = model.a_for_exact_gap(90.0, 0.0, D_REF_NM)
assert abs(a_sc_check - (model.L_nm + D_REF_NM)) < 1e-6
sc_energy = model.energy_components(a_sc_check, 90.0, 0.0)
assert abs(sc_energy["Dipole"]) < 1e-10

D_ref_gamma0 = model.nearest_exact_gap(
    REFERENCE["a_nm"], REFERENCE["alpha_deg"], 0.0
)

print("Reference calibration")
print("---------------------")
print(
    f"L={REFERENCE['L_nm']:.1f} nm, "
    f"a_ref={REFERENCE['a_nm']:.3f} nm, "
    f"alpha_ref={REFERENCE['alpha_deg']:.2f} deg, "
    f"gamma_ref={REFERENCE['gamma_deg']:.2f} deg relative to Q_SC"
)
print(f"D_ref exact Euclidean = {D_REF_NM:.6f} nm")
print(f"Legacy support-plane gap at same state = {D_REF_SUPPORT_NM:.6f} nm")
print(f"Sharp-cube VdW-body gap at same state = {reference_sharp_gap:.6f} nm")
print(f"Exact gap for gamma=0 SC frame at same a,alpha = {D_ref_gamma0:.6f} nm")
print(
    f"Reference interaction: Dip={reference_energy['Dipole']:.4f}, "
    f"VdW_face6={reference_energy['VdW']:.4f}, "
    f"VdW_full26={reference_energy['VdW_full26']:.4f}, "
    f"VdW_tail20={reference_energy['VdW_tail20']:.4f}, "
    f"sum={reference_energy['Interaction']:.4f} kBT"
)
print(
    f"SC check: a(90,gamma=0)=L+D_ref={a_sc_check:.6f} nm, "
    f"Dip={sc_energy['Dipole']:.3e}, "
    f"VdW_face6={sc_energy['VdW']:.4f} kBT"
)

In [ ]:
# ============================================================
#  ALPHA–GAMMA SCAN AT CONSTANT EXACT REFERENCE GAP
# ============================================================
alpha_regular_grid = np.linspace(
    SCAN["alpha_min_deg"],
    SCAN["alpha_max_deg"],
    SCAN["N_alpha"],
)
alphas = np.unique(np.sort(np.append(
    alpha_regular_grid,
    REFERENCE["alpha_deg"],
)))
gammas = np.linspace(-60.0, 60.0, SCAN["N_gamma"], endpoint=False)
gamma_step = 120.0 / SCAN["N_gamma"]

n_alpha, n_gamma = len(alphas), len(gammas)
A_grid = np.empty((n_alpha, n_gamma))
Eint_grid = np.empty_like(A_grid)
Evdw_grid = np.empty_like(A_grid)
Evdw_full26_grid = np.empty_like(A_grid)
Evdw_tail20_grid = np.empty_like(A_grid)
Edd_grid = np.empty_like(A_grid)

gamma_opt = np.empty(n_alpha)
a_opt = np.empty(n_alpha)
dip_opt = np.empty(n_alpha)
vdw_opt = np.empty(n_alpha)
vdw_full26_opt = np.empty(n_alpha)
vdw_tail20_opt = np.empty(n_alpha)
eint_opt = np.empty(n_alpha)
F_gamma = np.empty(n_alpha)
D_exact_opt = np.empty(n_alpha)
D_support_opt = np.empty(n_alpha)
D_sharp_opt = np.empty(n_alpha)

a_sc_frame = np.empty(n_alpha)
eint_sc_frame = np.empty(n_alpha)
a_legacy_frame = np.empty(n_alpha)
eint_legacy_frame = np.empty(n_alpha)

t0 = time.perf_counter()
for ia, alpha in enumerate(alphas):
    for ig, gamma in enumerate(gammas):
        a_nm, comp = model.state_at_gap(alpha, gamma, D_REF_NM)
        A_grid[ia, ig] = a_nm
        Eint_grid[ia, ig] = comp["Interaction"]
        Evdw_grid[ia, ig] = comp["VdW"]
        Evdw_full26_grid[ia, ig] = comp["VdW_full26"]
        Evdw_tail20_grid[ia, ig] = comp["VdW_tail20"]
        Edd_grid[ia, ig] = comp["Dipole"]

    best_grid_index = int(np.argmin(Eint_grid[ia]))
    best_grid_gamma = gammas[best_grid_index]

    if SCAN["refine_gamma_min"]:
        def objective(unwrapped_gamma):
            gamma = model.wrap_gamma(unwrapped_gamma)
            _, comp = model.state_at_gap(alpha, gamma, D_REF_NM)
            return comp["Interaction"]

        refined = minimize_scalar(
            objective,
            bounds=(
                best_grid_gamma - gamma_step,
                best_grid_gamma + gamma_step,
            ),
            method="bounded",
            options={"xatol": 2e-4, "maxiter": 80},
        )
        g_best = model.wrap_gamma(refined.x)
    else:
        g_best = best_grid_gamma

    # Mirror-related minima are physically equivalent. Select the negative
    # branch when degenerate so the plotted valley connects to gamma_ref=-15.
    _, comp_trial = model.state_at_gap(alpha, g_best, D_REF_NM)
    _, comp_mirror = model.state_at_gap(alpha, -g_best, D_REF_NM)
    if (
        abs(comp_mirror["Interaction"] - comp_trial["Interaction"]) < 1e-7
        and -g_best < g_best
    ):
        g_best = model.wrap_gamma(-g_best)

    a_best, comp_best = model.state_at_gap(alpha, g_best, D_REF_NM)
    gamma_opt[ia] = g_best
    a_opt[ia] = a_best
    dip_opt[ia] = comp_best["Dipole"]
    vdw_opt[ia] = comp_best["VdW"]
    vdw_full26_opt[ia] = comp_best["VdW_full26"]
    vdw_tail20_opt[ia] = comp_best["VdW_tail20"]
    eint_opt[ia] = comp_best["Interaction"]
    D_exact_opt[ia] = model.nearest_exact_gap(alpha_deg=alpha, a_nm=a_best, gamma_deg=g_best)
    D_support_opt[ia] = model.nearest_support_gap(alpha_deg=alpha, a_nm=a_best, gamma_deg=g_best)
    D_sharp_opt[ia] = model.nearest_sharp_gap(alpha_deg=alpha, a_nm=a_best, gamma_deg=g_best)

    # Per-particle coherent-domain free energy:
    # f_N = u_min - ln<exp[-N(u-u_min)]>/N.
    # N=1 is the local-rotor approximation; N->infinity gives min_gamma u.
    N_coh = float(SCAN["N_coherent_particles"])
    row_min = np.min(Eint_grid[ia])
    F_gamma[ia] = row_min - np.log(
        np.mean(np.exp(-N_coh * (Eint_grid[ia] - row_min)))
    ) / N_coh

    a0, c0 = model.state_at_gap(alpha, 0.0, D_REF_NM)
    a_sc_frame[ia] = a0
    eint_sc_frame[ia] = c0["Interaction"]

    aleg, cleg = model.state_at_gap(
        alpha, REFERENCE["gamma_deg"], D_REF_NM
    )
    a_legacy_frame[ia] = aleg
    eint_legacy_frame[ia] = cleg["Interaction"]

    if ia % 10 == 0 or ia == n_alpha - 1:
        print(
            f"alpha={alpha:5.1f} deg | "
            f"gamma_opt={g_best:7.2f} deg | "
            f"a={a_best:7.3f} nm | "
            f"Eint={comp_best['Interaction']:8.3f} kBT"
        )

elapsed = time.perf_counter() - t0
print(f"Scan completed in {elapsed:.1f} s.")

global_index = int(np.argmin(eint_opt))
free_index = int(np.argmin(F_gamma))

scan_table = np.column_stack([
    alphas,
    gamma_opt,
    a_opt,
    D_exact_opt,
    D_support_opt,
    dip_opt,
    vdw_opt,
    vdw_full26_opt,
    vdw_tail20_opt,
    eint_opt,
    F_gamma,
    D_sharp_opt,
    a_sc_frame,
    eint_sc_frame,
    a_legacy_frame,
    eint_legacy_frame,
])
header = (
    "alpha_deg,gamma_opt_deg,a_opt_nm,D_exact_nm,D_support_nm,"
    "Edd26_kBT,EVdW6_kBT,EVdW26_diag_kBT,EVdW_tail20_diag_kBT,"
    "Etotal_26plus6_kBT,F_gamma_26plus6_kBT,"
    "D_sharp_nm,a_SCframe_nm,Etotal_26plus6_SCframe_kBT,"
    "a_legacy_nm,Etotal_26plus6_legacyframe_kBT"
)
np.savetxt(
    DIAGNOSTIC_DIR / "V620_internal_gamma_relaxation_scan.csv",
    scan_table,
    delimiter=",",
    header=header,
    comments="",
)
np.savez_compressed(
    DIAGNOSTIC_DIR / "V620_internal_gamma_grid.npz",
    alphas=alphas,
    gammas=gammas,
    A_grid=A_grid,
    Eint_grid=Eint_grid,
    Evdw_grid=Evdw_grid,
    Evdw_full26_grid=Evdw_full26_grid,
    Evdw_tail20_grid=Evdw_tail20_grid,
    Edd_grid=Edd_grid,
    D_ref_nm=D_REF_NM,
    Hamiltonian=np.array("Edd26+EVdW6_primitive_face_bonds"),
    dipole_neighbours=np.int64(26),
    vdw_main_neighbours=np.int64(6),
    gap_constraint_neighbours=np.int64(26),
)

print("\nMinima")
print("------")
print(
    f"T=0 min: alpha={alphas[global_index]:.2f} deg, "
    f"gamma={gamma_opt[global_index]:.2f} deg, "
    f"a={a_opt[global_index]:.3f} nm, "
    f"Eint={eint_opt[global_index]:.4f} kBT"
)
print(
    f"F_gamma min: alpha={alphas[free_index]:.2f} deg, "
    f"F_gamma={F_gamma[free_index]:.4f} kBT "
    f"(N_coh={SCAN['N_coherent_particles']})"
)
if np.min(D_sharp_opt) <= 0.0:
    raise RuntimeError(
        "A state overlaps the sharp cubes used by the VdW quadrature."
    )
if np.max(np.abs(
    vdw_full26_opt - vdw_opt - vdw_tail20_opt
)) > 1e-10:
    raise RuntimeError("VdW shell decomposition identity failed.")
if np.max(np.abs(eint_opt - dip_opt - vdw_opt)) > 1e-10:
    raise RuntimeError("Interaction must equal Dipole26 + VdW6.")
high_alpha = alphas >= 80.0 - 1e-12
if np.any(np.diff(vdw_opt[high_alpha]) > 2e-6):
    raise RuntimeError("VdW6 must decrease monotonically from 80 to 90 deg.")
if abs(alphas[int(np.argmin(vdw_opt))] - 90.0) > 1e-12:
    raise RuntimeError("The six-face-bond VdW minimum must be at 90 deg.")
print(f"Minimum sharp-cube gap on relaxed path = {np.min(D_sharp_opt):.4f} nm")
print(
    "VdW6 check: monotonic from 80 to 90 deg; "
    f"global minimum at alpha={alphas[int(np.argmin(vdw_opt))]:.1f} deg."
)

In [ ]:
# ============================================================
#  V6.03-STYLE MULTI-SIZE DATA: L = 13, 16, 19.5 nm
# ============================================================
# All sizes use the same exact surface distance calibrated from the
# L=16 nm, alpha=74.0 deg reference state.  This is the explicit
# same-ligand-shell assumption; no reference lattice parameter is invented
# for L=13 nm, and the L=19.5 nm / alpha=60 deg point is kept as validation.

FIGURE_SIZES_NM = (16.0, 13.0, 19.5)
LANDSCAPE = {
    # Keep the original 0.5-degree alpha scan and resolve the outward
    # direction from the exact contact branch with 81 points.
    "N_alpha": SCAN["N_alpha"],
    "N_a": 81,
    "gamma_probe_points": 121,
    "gamma_xatol_deg": 0.06,
}


def size_tag(L_nm):
    text = f"{float(L_nm):g}"
    return text.replace(".", "p")


def best_state_on_reference_gap(model_i, alpha_deg, gamma_grid=gammas):
    '''Minimize Edd26+EVdW6 over gamma at D_exact(all 26)=D_ref.'''
    row = []
    for gamma_deg in gamma_grid:
        a_nm, comp = model_i.state_at_gap(alpha_deg, gamma_deg, D_REF_NM)
        row.append((comp["Interaction"], gamma_deg, a_nm, comp))

    _, gamma_seed, _, _ = min(row, key=lambda item: item[0])
    gamma_step_i = 120.0 / len(gamma_grid)

    def objective(gamma_unwrapped):
        gamma_deg = model_i.wrap_gamma(gamma_unwrapped)
        _, comp = model_i.state_at_gap(alpha_deg, gamma_deg, D_REF_NM)
        return comp["Interaction"]

    refined = minimize_scalar(
        objective,
        bounds=(gamma_seed - gamma_step_i, gamma_seed + gamma_step_i),
        method="bounded",
        options={"xatol": 2e-4, "maxiter": 80},
    )
    gamma_best = model_i.wrap_gamma(refined.x)

    # Choose the negative member of a mirror-degenerate pair, consistent
    # with the legacy reference gamma=-15 deg.
    a_trial, comp_trial = model_i.state_at_gap(
        alpha_deg, gamma_best, D_REF_NM
    )
    a_mirror, comp_mirror = model_i.state_at_gap(
        alpha_deg, -gamma_best, D_REF_NM
    )
    if (
        abs(comp_mirror["Interaction"] - comp_trial["Interaction"]) < 1e-7
        and -gamma_best < gamma_best
    ):
        gamma_best = model_i.wrap_gamma(-gamma_best)
        a_trial, comp_trial = a_mirror, comp_mirror

    if abs(gamma_best) < 1e-4:
        gamma_best = 0.0
    return a_trial, gamma_best, comp_trial


def scan_reference_gap_size(L_nm):
    '''Full alpha-gamma boundary scan for one particle size.'''
    model_i = ReferenceGapRhombohedralSL(L_nm)
    n_alpha_i, n_gamma_i = len(alphas), len(gammas)

    A_i = np.empty((n_alpha_i, n_gamma_i))
    E_i = np.empty_like(A_i)
    Edd_i = np.empty_like(A_i)
    Evdw_i = np.empty_like(A_i)
    Evdw_full26_i = np.empty_like(A_i)
    Evdw_tail20_i = np.empty_like(A_i)

    gamma_i = np.empty(n_alpha_i)
    a_i = np.empty(n_alpha_i)
    dip_i = np.empty(n_alpha_i)
    vdw_i = np.empty(n_alpha_i)
    vdw_full26_i = np.empty(n_alpha_i)
    vdw_tail20_i = np.empty(n_alpha_i)
    total_i = np.empty(n_alpha_i)
    exact_i = np.empty(n_alpha_i)
    sharp_i = np.empty(n_alpha_i)

    print(f"\nReference-gap scan for L={L_nm:g} nm")
    t_size = time.perf_counter()
    for ia, alpha_deg in enumerate(alphas):
        for ig, gamma_deg in enumerate(gammas):
            a_nm, comp = model_i.state_at_gap(
                alpha_deg, gamma_deg, D_REF_NM
            )
            A_i[ia, ig] = a_nm
            E_i[ia, ig] = comp["Interaction"]
            Edd_i[ia, ig] = comp["Dipole"]
            Evdw_i[ia, ig] = comp["VdW"]
            Evdw_full26_i[ia, ig] = comp["VdW_full26"]
            Evdw_tail20_i[ia, ig] = comp["VdW_tail20"]

        best_grid = int(np.argmin(E_i[ia]))
        gamma_seed = gammas[best_grid]

        def objective(gamma_unwrapped):
            gamma_deg = model_i.wrap_gamma(gamma_unwrapped)
            _, comp = model_i.state_at_gap(
                alpha_deg, gamma_deg, D_REF_NM
            )
            return comp["Interaction"]

        refined = minimize_scalar(
            objective,
            bounds=(gamma_seed - gamma_step, gamma_seed + gamma_step),
            method="bounded",
            options={"xatol": 2e-4, "maxiter": 80},
        )
        gamma_best = model_i.wrap_gamma(refined.x)

        a_best, comp_best = model_i.state_at_gap(
            alpha_deg, gamma_best, D_REF_NM
        )
        a_mirror, comp_mirror = model_i.state_at_gap(
            alpha_deg, -gamma_best, D_REF_NM
        )
        if (
            abs(
                comp_mirror["Interaction"]
                - comp_best["Interaction"]
            ) < 1e-7
            and -gamma_best < gamma_best
        ):
            gamma_best = model_i.wrap_gamma(-gamma_best)
            a_best, comp_best = a_mirror, comp_mirror

        if abs(gamma_best) < 1e-4:
            gamma_best = 0.0

        gamma_i[ia] = gamma_best
        a_i[ia] = a_best
        dip_i[ia] = comp_best["Dipole"]
        vdw_i[ia] = comp_best["VdW"]
        vdw_full26_i[ia] = comp_best["VdW_full26"]
        vdw_tail20_i[ia] = comp_best["VdW_tail20"]
        total_i[ia] = comp_best["Interaction"]
        exact_i[ia] = model_i.nearest_exact_gap(
            a_best, alpha_deg, gamma_best
        )
        sharp_i[ia] = model_i.nearest_sharp_gap(
            a_best, alpha_deg, gamma_best
        )

        if ia % 15 == 0 or ia == n_alpha_i - 1:
            print(
                f"  alpha={alpha_deg:5.1f} deg | "
                f"gamma={gamma_best:7.2f} deg | "
                f"a={a_best:7.3f} nm | "
                f"Eint={total_i[ia]:8.3f} kBT"
            )

    i_grid_min = int(np.argmin(total_i))

    # Refine alpha with the same full gamma minimization.  This is important
    # near the gamma=0 branch, where a three-point parabola can miss a shallow
    # cusp even though the 0.5-degree scan looks smooth.
    if 0 < i_grid_min < len(alphas) - 1:
        alpha_cache = {}

        def alpha_objective(alpha_deg):
            key = round(float(alpha_deg), 9)
            if key not in alpha_cache:
                alpha_cache[key] = best_state_on_reference_gap(
                    model_i, float(alpha_deg)
                )
            return alpha_cache[key][2]["Interaction"]

        alpha_refined = minimize_scalar(
            alpha_objective,
            bounds=(
                float(alphas[i_grid_min - 1]),
                float(alphas[i_grid_min + 1]),
            ),
            method="bounded",
            options={"xatol": 2e-4, "maxiter": 40},
        )
        alpha_star = float(alpha_refined.x)
    else:
        alpha_star = float(alphas[i_grid_min])

    a_star, gamma_star, comp_star = best_state_on_reference_gap(
        model_i, alpha_star
    )
    if comp_star["Interaction"] > total_i[i_grid_min] + 1e-6:
        alpha_star = float(alphas[i_grid_min])
        a_star = float(a_i[i_grid_min])
        gamma_star = float(gamma_i[i_grid_min])
        comp_star = {
            "Dipole": float(dip_i[i_grid_min]),
            "VdW": float(vdw_i[i_grid_min]),
            "VdW_face6": float(vdw_i[i_grid_min]),
            "VdW_full26": float(vdw_full26_i[i_grid_min]),
            "VdW_tail20": float(vdw_tail20_i[i_grid_min]),
            "Interaction": float(total_i[i_grid_min]),
        }

    if np.max(np.abs(exact_i - D_REF_NM)) > 5e-7:
        raise RuntimeError(f"Exact-gap residual failed for L={L_nm:g} nm.")
    if np.min(sharp_i) <= 0.0:
        raise RuntimeError(f"Sharp VdW cubes overlap for L={L_nm:g} nm.")
    if np.max(np.abs(
        vdw_full26_i - vdw_i - vdw_tail20_i
    )) > 1e-10:
        raise RuntimeError(
            f"VdW shell decomposition identity failed for L={L_nm:g} nm."
        )
    if np.max(np.abs(total_i - dip_i - vdw_i)) > 1e-10:
        raise RuntimeError(
            f"Interaction is not Edd26 + EVdW6 for L={L_nm:g} nm."
        )
    high_alpha_i = alphas >= 80.0 - 1e-12
    if np.any(np.diff(vdw_i[high_alpha_i]) > 2e-6):
        raise RuntimeError(
            f"VdW6 is not monotonic from 80 to 90 deg for L={L_nm:g} nm."
        )
    if abs(alphas[int(np.argmin(vdw_i))] - 90.0) > 1e-12:
        raise RuntimeError(
            f"The VdW6 minimum is not at 90 deg for L={L_nm:g} nm."
        )

    case = {
        "L_nm": float(L_nm),
        "model": model_i,
        "alphas": alphas.copy(),
        "gammas": gammas.copy(),
        "A_grid": A_i,
        "E_grid": E_i,
        "Edd_grid": Edd_i,
        "Evdw_grid": Evdw_i,
        "Evdw_full26_grid": Evdw_full26_i,
        "Evdw_tail20_grid": Evdw_tail20_i,
        "gamma_opt": gamma_i,
        "a_opt": a_i,
        "dipole": dip_i,
        "vdw": vdw_i,
        "vdw_full26": vdw_full26_i,
        "vdw_tail20": vdw_tail20_i,
        "interaction": total_i,
        "D_exact": exact_i,
        "D_sharp": sharp_i,
        "alpha_star": alpha_star,
        "gamma_star": float(gamma_star),
        "a_star": float(a_star),
        "comp_star": comp_star,
    }
    print(
        f"  minimum: alpha={alpha_star:.3f} deg, "
        f"gamma={gamma_star:.3f} deg, a={a_star:.4f} nm, "
        f"Edd={comp_star['Dipole']:.4f}, "
        f"VdW6={comp_star['VdW']:.4f}, "
        f"VdW26={comp_star['VdW_full26']:.4f}, "
        f"Total={comp_star['Interaction']:.4f} kBT"
    )
    print(
        f"  completed in {time.perf_counter() - t_size:.1f} s; "
        f"minimum sharp gap={np.min(sharp_i):.4f} nm"
    )
    return case


# Reuse the already evaluated L=16 nm grid exactly.
size_cases = {
    16.0: {
        "L_nm": 16.0,
        "model": model,
        "alphas": alphas.copy(),
        "gammas": gammas.copy(),
        "A_grid": A_grid.copy(),
        "E_grid": Eint_grid.copy(),
        "Edd_grid": Edd_grid.copy(),
        "Evdw_grid": Evdw_grid.copy(),
        "Evdw_full26_grid": Evdw_full26_grid.copy(),
        "Evdw_tail20_grid": Evdw_tail20_grid.copy(),
        "gamma_opt": gamma_opt.copy(),
        "a_opt": a_opt.copy(),
        "dipole": dip_opt.copy(),
        "vdw": vdw_opt.copy(),
        "vdw_full26": vdw_full26_opt.copy(),
        "vdw_tail20": vdw_tail20_opt.copy(),
        "interaction": eint_opt.copy(),
        "D_exact": D_exact_opt.copy(),
        "D_sharp": D_sharp_opt.copy(),
    }
}

for L_nm in (13.0, 19.5):
    size_cases[L_nm] = scan_reference_gap_size(L_nm)

# Add the fully nested alpha/gamma minimum to the reused L=16 nm case.
case16 = size_cases[16.0]
i16 = int(np.argmin(case16["interaction"]))
if 0 < i16 < len(alphas) - 1:
    alpha16_cache = {}

    def alpha16_objective(alpha_deg):
        key = round(float(alpha_deg), 9)
        if key not in alpha16_cache:
            alpha16_cache[key] = best_state_on_reference_gap(
                model, float(alpha_deg)
            )
        return alpha16_cache[key][2]["Interaction"]

    alpha16_refined = minimize_scalar(
        alpha16_objective,
        bounds=(
            float(alphas[i16 - 1]),
            float(alphas[i16 + 1]),
        ),
        method="bounded",
        options={"xatol": 2e-4, "maxiter": 40},
    )
    alpha16_star = float(alpha16_refined.x)
else:
    alpha16_star = float(alphas[i16])
a16_star, gamma16_star, comp16_star = best_state_on_reference_gap(
    model, alpha16_star
)
if comp16_star["Interaction"] > case16["interaction"][i16] + 1e-6:
    alpha16_star = float(alphas[i16])
    a16_star = float(case16["a_opt"][i16])
    gamma16_star = float(case16["gamma_opt"][i16])
    comp16_star = {
        "Dipole": float(case16["dipole"][i16]),
        "VdW": float(case16["vdw"][i16]),
        "VdW_face6": float(case16["vdw"][i16]),
        "VdW_full26": float(case16["vdw_full26"][i16]),
        "VdW_tail20": float(case16["vdw_tail20"][i16]),
        "Interaction": float(case16["interaction"][i16]),
    }
case16.update({
    "alpha_star": alpha16_star,
    "gamma_star": float(gamma16_star),
    "a_star": float(a16_star),
    "comp_star": comp16_star,
})

# Attach the complete Hamiltonian to every size without changing the
# already-validated interaction minimization.  For the present orientation
# Q(gamma)=R_x(gamma)Q_SC with the field along x, Zeeman and cubic anisotropy
# are independent of a, alpha, and gamma.  The assertions below make that
# property explicit and prevent a future model change from silently turning
# this into an invalid constant shift.
for case in size_cases.values():
    model_i = case["model"]
    comp_star_full = model_i.energy_components(
        case["a_star"], case["alpha_star"], case["gamma_star"]
    )
    offset_samples = []
    for probe_i in (0, len(case["alphas"]) // 2, -1):
        comp_probe = model_i.energy_components(
            case["a_opt"][probe_i],
            case["alphas"][probe_i],
            case["gamma_opt"][probe_i],
        )
        offset_samples.append(
            comp_probe["Total"] - comp_probe["Interaction"]
        )
        if abs(
            comp_probe["Total"]
            - comp_probe["Interaction"]
            - comp_probe["Zeeman"]
            - comp_probe["Anisotropy"]
        ) > 1e-10:
            raise RuntimeError("Complete Total energy decomposition failed.")

    if np.ptp(offset_samples) > 1e-10:
        raise RuntimeError(
            "Zeeman + anisotropy is not constant over the current scan; "
            "the optimization objective must then be changed to Total."
        )

    case["comp_star"] = comp_star_full
    case["zeeman"] = float(comp_star_full["Zeeman"])
    case["anisotropy"] = float(comp_star_full["Anisotropy"])
    case["single_particle_offset"] = float(
        comp_star_full["Zeeman"] + comp_star_full["Anisotropy"]
    )
    case["total"] = (
        np.asarray(case["interaction"])
        + case["single_particle_offset"]
    )
    if not np.allclose(
        case["total"],
        np.asarray(case["interaction"])
        + case["zeeman"]
        + case["anisotropy"],
        atol=1e-10,
        rtol=0.0,
    ):
        raise RuntimeError("Full Total array assembly failed.")
    if int(np.argmin(case["total"])) != int(
        np.argmin(case["interaction"])
    ):
        raise RuntimeError(
            "Adding the single-particle terms changed the alpha minimum."
        )


def feasible_gamma_intervals(model_i, a_nm, alpha_deg):
    '''Feasible intervals in gamma in [0,60] for D_exact >= D_ref.'''
    probe = np.linspace(
        0.0, 60.0, LANDSCAPE["gamma_probe_points"]
    )
    f = np.array([
        model_i.nearest_exact_gap(a_nm, alpha_deg, gamma_deg)
        - D_REF_NM
        for gamma_deg in probe
    ])
    tol = 2e-8
    feasible = f >= -tol
    intervals = []

    if np.any(feasible):
        indices = np.flatnonzero(feasible)
        split_at = np.where(np.diff(indices) > 1)[0]
        blocks = np.split(indices, split_at + 1)
        for block in blocks:
            left_i, right_i = int(block[0]), int(block[-1])
            if left_i == 0:
                left = 0.0
            else:
                left = brentq(
                    lambda g: (
                        model_i.nearest_exact_gap(a_nm, alpha_deg, g)
                        - D_REF_NM
                    ),
                    probe[left_i - 1],
                    probe[left_i],
                    xtol=1e-8,
                )
            if right_i == len(probe) - 1:
                right = 60.0
            else:
                right = brentq(
                    lambda g: (
                        model_i.nearest_exact_gap(a_nm, alpha_deg, g)
                        - D_REF_NM
                    ),
                    probe[right_i],
                    probe[right_i + 1],
                    xtol=1e-8,
                )
            intervals.append((float(left), float(right)))
        return intervals

    # At a tangential hard-wall contact the feasible set can be narrower
    # than the geometry probe.  Locate the maximum gap without an energy
    # evaluation and keep the isolated contact point when it is physical.
    j = int(np.argmax(f))
    left = probe[max(j - 1, 0)]
    right = probe[min(j + 1, len(probe) - 1)]
    if right > left:
        peak = minimize_scalar(
            lambda g: -model_i.nearest_exact_gap(
                a_nm, alpha_deg, g
            ),
            bounds=(left, right),
            method="bounded",
            options={"xatol": 1e-7, "maxiter": 60},
        )
        peak_gap_residual = -float(peak.fun) - D_REF_NM
        if peak_gap_residual >= -tol:
            intervals.append((float(peak.x), float(peak.x)))
    return intervals


def relaxed_state_at_fixed_a(model_i, a_nm, alpha_deg):
    '''Minimize interaction over every feasible gamma interval.'''
    intervals = feasible_gamma_intervals(model_i, a_nm, alpha_deg)
    if not intervals:
        return np.nan, None

    cache = {}

    def evaluate(gamma_deg):
        key = round(float(gamma_deg), 10)
        if key not in cache:
            cache[key] = model_i.energy_components(
                a_nm, alpha_deg, gamma_deg
            )
        return cache[key]

    candidates = []
    for left, right in intervals:
        candidates.extend([left, right])
        if right - left > 2e-4:
            refined = minimize_scalar(
                lambda g: evaluate(g)["Interaction"],
                bounds=(left, right),
                method="bounded",
                options={
                    "xatol": LANDSCAPE["gamma_xatol_deg"],
                    "maxiter": 30,
                },
            )
            candidates.append(float(refined.x))

    best_gamma = None
    best_comp = None
    for gamma_deg in candidates:
        if (
            model_i.nearest_exact_gap(a_nm, alpha_deg, gamma_deg)
            < D_REF_NM - 5e-7
        ):
            continue
        comp = evaluate(gamma_deg)
        if (
            best_comp is None
            or comp["Interaction"] < best_comp["Interaction"]
        ):
            best_gamma = float(gamma_deg)
            best_comp = comp
    return best_gamma, best_comp


def make_fixed_alpha_scan(case):
    '''E(a) at the fully relaxed alpha minimum.'''
    model_i = case["model"]
    alpha_star = case["alpha_star"]

    contact_candidates = np.array([
        model_i.a_for_exact_gap(alpha_star, gamma_deg, D_REF_NM)
        for gamma_deg in gammas
    ])
    a_low = max(
        0.5,
        np.floor((np.min(contact_candidates) - 0.5) * 2.0) / 2.0,
    )
    a_high = np.ceil((case["a_star"] + 7.0) * 2.0) / 2.0
    a_values = np.unique(np.sort(np.append(
        np.linspace(a_low, a_high, 111),
        case["a_star"],
    )))

    dip = np.full_like(a_values, np.nan)
    vdw = np.full_like(a_values, np.nan)
    vdw_full26 = np.full_like(a_values, np.nan)
    vdw_tail20 = np.full_like(a_values, np.nan)
    interaction = np.full_like(a_values, np.nan)
    total = np.full_like(a_values, np.nan)
    gamma_relaxed = np.full_like(a_values, np.nan)
    gap = np.full_like(a_values, np.nan)

    for i, a_nm in enumerate(a_values):
        gamma_best, comp = relaxed_state_at_fixed_a(
            model_i, a_nm, alpha_star
        )
        if comp is None:
            continue
        gamma_relaxed[i] = -abs(gamma_best)
        dip[i] = comp["Dipole"]
        vdw[i] = comp["VdW"]
        vdw_full26[i] = comp["VdW_full26"]
        vdw_tail20[i] = comp["VdW_tail20"]
        interaction[i] = comp["Interaction"]
        total[i] = comp["Total"]
        gap[i] = model_i.nearest_exact_gap(
            a_nm, alpha_star, gamma_best
        )

    i_star_a = int(np.argmin(np.abs(a_values - case["a_star"])))
    if (
        not np.isfinite(interaction[i_star_a])
        or abs(
            interaction[i_star_a]
            - case["comp_star"]["Interaction"]
        ) > 0.03
    ):
        raise RuntimeError(
            f"E(a) does not recover the boundary minimum for "
            f"L={case['L_nm']:g} nm."
        )
    finite = np.isfinite(interaction)
    if np.max(np.abs(
        vdw_full26[finite] - vdw[finite] - vdw_tail20[finite]
    )) > 1e-10:
        raise RuntimeError(
            f"VdW shell decomposition failed in E(a) for "
            f"L={case['L_nm']:g} nm."
        )
    if np.max(np.abs(
        interaction[finite] - dip[finite] - vdw[finite]
    )) > 1e-10:
        raise RuntimeError(
            f"E(a) interaction is not Edd26 + EVdW6 for "
            f"L={case['L_nm']:g} nm."
        )
    if np.max(np.abs(
        total[finite]
        - interaction[finite]
        - case["zeeman"]
        - case["anisotropy"]
    )) > 1e-10:
        raise RuntimeError(
            f"E(a) Total does not include Zeeman and anisotropy for "
            f"L={case['L_nm']:g} nm."
        )
    return {
        "a": a_values,
        "dipole": dip,
        "vdw": vdw,
        "vdw_full26": vdw_full26,
        "vdw_tail20": vdw_tail20,
        "interaction": interaction,
        "total": total,
        "gamma": gamma_relaxed,
        "D_exact": gap,
    }


def make_relaxed_landscape(case):
    '''Gamma-relaxed E(a,alpha) on a contact-aligned curvilinear mesh.'''
    model_i = case["model"]
    alpha_values = np.asarray(case["alphas"]).copy()
    a_contact = np.asarray(case["a_opt"]).copy()
    gamma_contact = np.asarray(case["gamma_opt"]).copy()

    a_high_target = np.max(case["A_grid"]) + 5.0
    if case["L_nm"] in EXP_DATA:
        a_high_target = max(
            a_high_target,
            EXP_DATA[case["L_nm"]]["d"] + 1.0,
        )
    a_high = np.ceil(a_high_target * 2.0) / 2.0
    outward_fraction = np.linspace(0.0, 1.0, LANDSCAPE["N_a"])
    a_mesh = (
        a_contact[:, None]
        + outward_fraction[None, :]
        * (a_high - a_contact[:, None])
    )
    alpha_mesh = np.broadcast_to(
        alpha_values[:, None], a_mesh.shape
    ).copy()

    Einteraction = np.full_like(a_mesh, np.nan)
    Etotal = np.full_like(a_mesh, np.nan)
    gamma_relaxed = np.full_like(a_mesh, np.nan)
    exact_gap = np.full_like(a_mesh, np.nan)

    print(
        f"\nGamma-relaxed E(a,alpha) landscape for "
        f"L={case['L_nm']:g} nm: "
        f"{len(alpha_values)} x {LANDSCAPE['N_a']} "
        f"(contact-aligned)"
    )
    t_landscape = time.perf_counter()
    for ia, alpha_deg in enumerate(alpha_values):
        # The first column is exactly the already optimized D=D_ref branch.
        # Seeding it directly avoids losing a tangential hard-wall point in
        # the feasibility root finder and guarantees that the filled domain
        # touches the blue contact line without a masked-grid strip.
        Einteraction[ia, 0] = case["interaction"][ia]
        Etotal[ia, 0] = case["total"][ia]
        gamma_relaxed[ia, 0] = gamma_contact[ia]
        exact_gap[ia, 0] = case["D_exact"][ia]

        for ja in range(1, LANDSCAPE["N_a"]):
            a_nm = a_mesh[ia, ja]
            gamma_best, comp = relaxed_state_at_fixed_a(
                model_i, a_nm, alpha_deg
            )
            if comp is None:
                raise RuntimeError(
                    f"Unexpected infeasible point outside the contact "
                    f"branch at L={case['L_nm']:g} nm, "
                    f"alpha={alpha_deg:.3f} deg, a={a_nm:.6f} nm."
                )
            Einteraction[ia, ja] = comp["Interaction"]
            Etotal[ia, ja] = comp["Total"]
            gamma_relaxed[ia, ja] = -abs(gamma_best)
            exact_gap[ia, ja] = model_i.nearest_exact_gap(
                a_nm, alpha_deg, gamma_best
            )
        if ia % 10 == 0 or ia == len(alpha_values) - 1:
            print(
                f"  row {ia + 1:2d}/{len(alpha_values)} "
                f"(alpha={alpha_deg:5.1f} deg)"
            )

    if not np.all(np.isfinite(Etotal)):
        raise RuntimeError(
            f"Non-finite physical landscape state for L={case['L_nm']:g} nm."
        )
    if np.max(np.abs(a_mesh[:, 0] - a_contact)) > 1e-12:
        raise RuntimeError("Landscape boundary is not the blue contact line.")
    if np.max(np.abs(
        Einteraction[:, 0] - case["interaction"]
    )) > 1e-12:
        raise RuntimeError("Landscape boundary energy mismatch.")
    if np.min(exact_gap - D_REF_NM) < -5e-7:
        raise RuntimeError("Landscape violates the exact-gap constraint.")
    if np.max(np.abs(
        Etotal
        - Einteraction
        - case["zeeman"]
        - case["anisotropy"]
    )) > 1e-10:
        raise RuntimeError(
            "Landscape Total does not include Zeeman and anisotropy."
        )

    i_flat = int(np.argmin(Einteraction))
    i_land, j_land = np.unravel_index(i_flat, Einteraction.shape)
    if (
        Einteraction[i_land, j_land]
        < case["comp_star"]["Interaction"] - 0.08
    ):
        raise RuntimeError(
            f"Landscape is lower than the reference-gap boundary scan "
            f"for L={case['L_nm']:g} nm."
        )
    print(
        f"  landscape completed in "
        f"{time.perf_counter() - t_landscape:.1f} s"
    )
    return {
        "alphas": alpha_values,
        "alpha_mesh": alpha_mesh,
        "a_mesh": a_mesh,
        "a_contact": a_contact,
        "outward_fraction": outward_fraction,
        "interaction": Einteraction,
        "total": Etotal,
        "gamma": gamma_relaxed,
        "D_exact": exact_gap,
    }


# Calculate the V6.03-style slices and landscapes, then save all numerical
# data independently for the three sizes.
minima_rows = []
for L_nm in FIGURE_SIZES_NM:
    case = size_cases[L_nm]
    case["a_scan"] = make_fixed_alpha_scan(case)
    case["landscape"] = make_relaxed_landscape(case)
    tag = size_tag(L_nm)

    alpha_table = np.column_stack([
        case["alphas"],
        case["gamma_opt"],
        case["a_opt"],
        case["D_exact"],
        case["D_sharp"],
        case["dipole"],
        case["vdw"],
        case["vdw_full26"],
        case["vdw_tail20"],
        case["interaction"],
        np.full_like(case["alphas"], case["zeeman"]),
        np.full_like(case["alphas"], case["anisotropy"]),
        case["total"],
    ])
    np.savetxt(
        OUTPUT_DIR / f"V620_L{tag}_alpha_scan.csv",
        alpha_table,
        delimiter=",",
        header=(
            "alpha_deg,gamma_opt_deg,a_opt_nm,D_exact_nm,D_sharp_nm,"
            "Edd26_kBT,EVdW6_kBT,EVdW26_diag_kBT,"
            "EVdW_tail20_diag_kBT,Einteraction_26plus6_kBT,"
            "EZeeman_kBT,EAnisotropy_kBT,Etotal_kBT"
        ),
        comments="",
    )

    a_scan = case["a_scan"]
    np.savetxt(
        OUTPUT_DIR / f"V620_L{tag}_a_scan.csv",
        np.column_stack([
            a_scan["a"],
            a_scan["gamma"],
            a_scan["D_exact"],
            a_scan["dipole"],
            a_scan["vdw"],
            a_scan["vdw_full26"],
            a_scan["vdw_tail20"],
            a_scan["interaction"],
            np.full_like(a_scan["a"], case["zeeman"]),
            np.full_like(a_scan["a"], case["anisotropy"]),
            a_scan["total"],
        ]),
        delimiter=",",
        header=(
            "a_nm,gamma_opt_deg,D_exact_nm,"
            "Edd26_kBT,EVdW6_kBT,EVdW26_diag_kBT,"
            "EVdW_tail20_diag_kBT,Einteraction_26plus6_kBT,"
            "EZeeman_kBT,EAnisotropy_kBT,Etotal_kBT"
        ),
        comments="",
    )

    land = case["landscape"]
    np.savez_compressed(
        OUTPUT_DIR / f"V620_L{tag}_relaxed_landscape.npz",
        alphas=land["alphas"],
        alpha_mesh_deg=land["alpha_mesh"],
        a_mesh_nm=land["a_mesh"],
        a_contact_nm=land["a_contact"],
        outward_fraction=land["outward_fraction"],
        Einteraction_26plus6_kBT=land["interaction"],
        Etotal_kBT=land["total"],
        EZeeman_kBT=np.float64(case["zeeman"]),
        EAnisotropy_kBT=np.float64(case["anisotropy"]),
        gamma_opt=land["gamma"],
        D_exact_nm=land["D_exact"],
        D_ref_nm=D_REF_NM,
        L_nm=L_nm,
        Hamiltonian=np.array(
            "Zeeman+Anisotropy+Edd26+EVdW6_primitive_face_bonds"
        ),
        dipole_neighbours=np.int64(26),
        vdw_main_neighbours=np.int64(6),
        gap_constraint_neighbours=np.int64(26),
    )

    minima_rows.append([
        L_nm,
        case["alpha_star"],
        case["gamma_star"],
        case["a_star"],
        case["comp_star"]["Dipole"],
        case["comp_star"]["VdW"],
        case["comp_star"]["VdW_full26"],
        case["comp_star"]["VdW_tail20"],
        case["comp_star"]["Interaction"],
        case["comp_star"]["Zeeman"],
        case["comp_star"]["Anisotropy"],
        case["comp_star"]["Total"],
    ])

np.savetxt(
    OUTPUT_DIR / "V620_size_minima.csv",
    np.asarray(minima_rows),
    delimiter=",",
    header=(
        "L_nm,alpha_min_deg,gamma_min_deg,a_min_nm,"
        "Edd26_kBT,EVdW6_kBT,EVdW26_diag_kBT,"
        "EVdW_tail20_diag_kBT,Einteraction_26plus6_kBT,"
        "EZeeman_kBT,EAnisotropy_kBT,Etotal_kBT"
    ),
    comments="",
)

In [ ]:
# ============================================================
#  V6.03-STYLE MULTI-SIZE FIGURES
# ============================================================
KW_DIP = dict(
    color="royalblue", lw=2.0, ls="-", label="Dipole"
)
KW_VDW = dict(
    color="tomato", lw=2.0, ls="--", label="VdW"
)
KW_TOTAL = dict(
    color="black", lw=2.5, ls="-", label="Total"
)


def set_component_ylim(ax, dipole, vdw, total):
    values = np.concatenate([
        np.asarray(dipole)[np.isfinite(dipole)],
        np.asarray(vdw)[np.isfinite(vdw)],
        np.asarray(total)[np.isfinite(total)],
    ])
    if len(values):
        lo, hi = float(np.min(values)), float(np.max(values))
        pad = 0.12 * (hi - lo) if hi > lo else 1.0
        ax.set_ylim(lo - pad, hi + pad)


def draw_component_curves(
    ax, x, dipole, vdw, sum_label="Total"
):
    # In every E-vs-alpha and E-vs-a component panel, Total is defined
    # strictly as the sum of the two displayed interaction components.
    total = np.asarray(dipole) + np.asarray(vdw)
    ax.plot(x, dipole, **KW_DIP)
    ax.plot(x, vdw, **KW_VDW)
    total_style = KW_TOTAL.copy()
    total_style["label"] = sum_label
    ax.plot(x, total, **total_style)
    ax.axhline(0.0, color="gray", lw=0.8)
    ax.grid(True, alpha=0.3)
    set_component_ylim(ax, dipole, vdw, total)


def save_pdf_png(fig, stem):
    fig.savefig(
        FIGURE_DIR / f"{stem}.pdf",
        dpi=600,
        bbox_inches="tight",
    )
    fig.savefig(
        FIGURE_DIR / f"{stem}.png",
        dpi=600,
        bbox_inches="tight",
    )


def add_experimental_alpha(ax, case):
    '''Add only the requested alpha_exp annotation to an energy curve.'''
    if case["L_nm"] not in EXP_DATA:
        return None
    return ax.axvline(
        EXP_DATA[case["L_nm"]]["alpha"],
        color="gray",
        ls=":",
        lw=1.4,
        label=r"$\alpha_{\rm exp}$",
    )


def plot_v603_landscape(case):
    '''Restore the detailed V6.03-style E(a,alpha) multi-panel figure.'''
    L_nm = case["L_nm"]
    tag = size_tag(L_nm)
    land = case["landscape"]
    AA = land["a_mesh"]
    AL = land["alpha_mesh"]
    Eshow = land["total"]
    finite_E = Eshow[np.isfinite(Eshow)]
    lo = float(np.min(finite_E))
    hi = float(np.percentile(finite_E, 98.0))
    if hi <= lo:
        hi = lo + 1.0
    levels = np.linspace(lo, hi, 41)

    fig = plt.figure(figsize=(7.0, 5.75))
    gs = fig.add_gridspec(
        2,
        2,
        left=0.10,
        right=0.97,
        bottom=0.10,
        top=0.88,
        hspace=0.48,
        wspace=0.40,
    )
    ax_land = fig.add_subplot(gs[0, 0])
    ax_key = fig.add_subplot(gs[0, 1])
    ax_alpha = fig.add_subplot(gs[1, 0])
    ax_a = fig.add_subplot(gs[1, 1])
    ax_key.axis("off")

    cf = ax_land.contourf(
        AA,
        AL,
        Eshow,
        levels=levels,
        cmap="RdYlBu_r",
        extend="max",
        antialiased=False,
    )
    cb = fig.colorbar(
        cf,
        ax=ax_land,
        format="%.1f",
        fraction=0.055,
        pad=0.035,
    )
    cb.set_ticks(np.linspace(lo, hi, 5))
    cb.ax.yaxis.set_major_formatter(
        mpl.ticker.FormatStrFormatter("%.1f")
    )
    cb.set_label(r"$E_{\rm total}/k_BT$", labelpad=4)
    cb.ax.tick_params(labelsize=7)
    ax_land.contour(
        AA,
        AL,
        Eshow,
        levels=16,
        colors="white",
        linewidths=0.45,
        alpha=0.55,
    )

    line_a = ax_land.axvline(
        case["a_star"],
        color="orange",
        ls="--",
        lw=1.8,
        zorder=10,
        label=rf"$a = {case['a_star']:.2f}$ nm",
    )
    line_alpha = ax_land.axhline(
        case["alpha_star"],
        color="darkturquoise",
        ls="--",
        lw=1.8,
        zorder=10,
        label=rf"$\alpha = {case['alpha_star']:.1f}^\circ$",
    )
    line_gap, = ax_land.plot(
        land["a_contact"],
        land["alphas"],
        color="dodgerblue",
        ls="--",
        lw=1.8,
        alpha=0.95,
        zorder=11,
        label=rf"$D = {D_DISPLAY_NM:.1f}$ nm",
    )
    sim, = ax_land.plot(
        case["a_star"],
        case["alpha_star"],
        marker="*",
        ls="None",
        ms=10.5,
        mfc="gold",
        mec="black",
        mew=0.8,
        zorder=100,
        clip_on=False,
        label=(
            rf"Sim ({case['a_star']:.2f} nm, "
            rf"{case['alpha_star']:.1f}°)"
        ),
    )
    legend_handles = [sim]
    if L_nm in EXP_DATA:
        exp = EXP_DATA[L_nm]
        d_decimals = (
            1
            if abs(exp["d"] - round(exp["d"], 1)) < 1e-10
            else 2
        )
        exp_handle, = ax_land.plot(
            exp["d"],
            exp["alpha"],
            marker="^",
            ls="None",
            ms=7.0,
            mfc="limegreen",
            mec="black",
            mew=0.8,
            zorder=101,
            clip_on=False,
            label=(
                rf"Exp ({exp['d']:.{d_decimals}f} nm, "
                rf"{exp['alpha']:.1f}°)"
            ),
        )
        legend_handles.append(exp_handle)
    legend_handles.extend([line_a, line_alpha, line_gap])

    ax_land.set_xlabel(r"$a$ (nm)")
    ax_land.set_ylabel(r"$\alpha$ (°)")
    ax_land.set_ylim(60.0, 90.0)
    ax_land.set_title(r"$E(a,\alpha)$ landscape")
    landscape_legend = ax_key.legend(
        handles=legend_handles,
        fontsize=7,
        loc="upper left",
        bbox_to_anchor=(0.02, 0.98),
        frameon=False,
        handlelength=2.0,
        borderaxespad=0.0,
    )
    ax_key.add_artist(landscape_legend)

    draw_component_curves(
        ax_alpha,
        case["alphas"],
        case["dipole"],
        case["vdw"],
        sum_label="Dipole + VdW",
    )
    add_experimental_alpha(ax_alpha, case)
    ax_alpha.set_xlim(58.5, 91.5)
    ax_alpha.set_xlabel(r"$\alpha$ (°)")
    ax_alpha.set_ylabel(r"Energy ($k_BT$)")
    ax_alpha.set_title(
        rf"$E$ vs $\alpha$ at $D = {D_DISPLAY_NM:.1f}$ nm"
    )

    a_scan = case["a_scan"]
    draw_component_curves(
        ax_a,
        a_scan["a"],
        a_scan["dipole"],
        a_scan["vdw"],
    )
    ax_a.axvline(
        case["a_star"],
        color="gray",
        ls=":",
        lw=1.4,
        label="_nolegend_",
    )
    ax_a.set_xlabel(r"$a$ (nm)")
    ax_a.set_ylabel(r"Energy ($k_BT$)")
    ax_a.set_title(
        rf"$E$ vs $a$ at $\alpha = {case['alpha_star']:.1f}^\circ$"
    )

    alpha_handles, alpha_labels = ax_alpha.get_legend_handles_labels()
    alpha_legend = ax_key.legend(
        alpha_handles,
        alpha_labels,
        loc="center left",
        bbox_to_anchor=(0.02, 0.29),
        ncol=2,
        frameon=True,
        facecolor="white",
        edgecolor="0.6",
        framealpha=1.0,
        handlelength=2.2,
        columnspacing=1.1,
        borderaxespad=0.0,
    )
    ax_key.add_artist(alpha_legend)

    a_handles, a_labels = ax_a.get_legend_handles_labels()
    ax_key.legend(
        a_handles,
        a_labels,
        loc="lower left",
        bbox_to_anchor=(0.02, 0.01),
        ncol=3,
        frameon=True,
        facecolor="white",
        edgecolor="0.6",
        framealpha=1.0,
        handlelength=1.8,
        columnspacing=0.8,
        borderaxespad=0.0,
    )

    fig.suptitle(
        rf"V6.20 — V6.03-style $E(a,\alpha)$ landscape "
        rf"| $r_{{\rm round}}={PHYS['roundness_nm']:.1f}$ nm"
    )
    save_pdf_png(
        fig,
        f"V620_L{tag}_Fig2_E_vs_a_alpha",
    )
    plt.show()


def plot_v603_components(case):
    '''Restore the detailed V6.03-style component/geometry figure.'''
    L_nm = case["L_nm"]
    tag = size_tag(L_nm)
    a_scan = case["a_scan"]

    fig = plt.figure(figsize=(7.0, 5.75))
    gs = fig.add_gridspec(
        2,
        2,
        left=0.10,
        right=0.97,
        bottom=0.10,
        top=0.88,
        hspace=0.48,
        wspace=0.42,
    )
    ax_a = fig.add_subplot(gs[0, 0])
    ax_geometry = fig.add_subplot(gs[0, 1])
    ax_alpha = fig.add_subplot(gs[1, 0])
    ax_key = fig.add_subplot(gs[1, 1])
    ax_key.axis("off")

    draw_component_curves(
        ax_a,
        a_scan["a"],
        a_scan["dipole"],
        a_scan["vdw"],
    )
    ax_a.axvline(
        case["a_star"],
        color="gray",
        ls=":",
        lw=1.4,
        label="_nolegend_",
    )
    ax_a.set_xlabel(r"$a$ (nm)")
    ax_a.set_ylabel(r"Energy ($k_BT$)")
    ax_a.set_title(
        rf"$E$ vs $a$, $\alpha={case['alpha_star']:.1f}^\circ$"
    )

    ax_geometry.plot(
        case["alphas"],
        case["a_opt"],
        color="mediumpurple",
        lw=2.2,
        label=rf"$D = {D_DISPLAY_NM:.1f}$ nm",
    )
    ax_geometry.axvline(
        case["alpha_star"],
        color="gray",
        ls=":",
        lw=1.4,
        label=rf"$\alpha_{{\rm opt}}={case['alpha_star']:.1f}^\circ$",
    )
    ax_geometry.set_xlim(58.5, 91.5)
    ax_geometry.set_xlabel(r"$\alpha$ (°)")
    ax_geometry.set_ylabel(r"$a$ (nm)")
    ax_geometry.set_title("Constant-gap geometry")
    ax_geometry.grid(True, alpha=0.3)
    ax_geometry.legend(loc="best", frameon=True)

    draw_component_curves(
        ax_alpha,
        case["alphas"],
        case["dipole"],
        case["vdw"],
        sum_label="Dipole + VdW",
    )
    add_experimental_alpha(ax_alpha, case)
    ax_alpha.set_xlim(58.5, 91.5)
    ax_alpha.set_xlabel(r"$\alpha$ (°)")
    ax_alpha.set_ylabel(r"Energy ($k_BT$)")
    ax_alpha.set_title(
        rf"$E$ vs $\alpha$, $D = {D_DISPLAY_NM:.1f}$ nm"
    )

    a_handles, a_labels = ax_a.get_legend_handles_labels()
    a_legend = ax_key.legend(
        a_handles,
        a_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.82),
        ncol=1,
        frameon=True,
        facecolor="white",
        edgecolor="0.6",
        framealpha=1.0,
        handlelength=2.4,
    )
    ax_key.add_artist(a_legend)

    alpha_handles, alpha_labels = ax_alpha.get_legend_handles_labels()
    ax_key.legend(
        alpha_handles,
        alpha_labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.18),
        ncol=1,
        frameon=True,
        facecolor="white",
        edgecolor="0.6",
        framealpha=1.0,
        handlelength=2.4,
    )

    fig.suptitle(
        rf"V6.20 — $E_{{dd}}$, $E_{{\rm VdW}}$, and "
        rf"$E_{{\rm total}}$ | $D = {D_DISPLAY_NM:.1f}$ nm, "
        rf"$r_{{\rm round}}={PHYS['roundness_nm']:.1f}$ nm"
    )
    save_pdf_png(
        fig,
        f"V620_L{tag}_Fig3b_Edd_EvdW_Etotal",
    )
    plt.show()


for L_nm in FIGURE_SIZES_NM:
    plot_v603_landscape(size_cases[L_nm])
    plot_v603_components(size_cases[L_nm])

print("\nV6.03-style multi-size outputs")
print("------------------------------")
for row in minima_rows:
    print(
        f"L={row[0]:4.1f} nm | alpha={row[1]:7.3f} deg | "
        f"gamma={row[2]:7.3f} deg | a={row[3]:8.4f} nm | "
        f"Edd26={row[4]:8.4f} | EVdW6={row[5]:8.4f} | "
        f"EVdW26diag={row[6]:8.4f} | "
        f"Interaction={row[8]:8.4f} | "
        f"Zeeman={row[9]:9.4f} | Anisotropy={row[10]:8.4f} | "
        f"Total={row[11]:9.4f} kBT"
    )
print(
    "\nLandscape: Etotal = Zeeman + Anisotropy + Edd26 + EVdW6. "
    "E-vs-alpha and E-vs-a curves: Total = Edd26 + EVdW6 only. "
    "EVdW26 and tail20 remain diagnostic only."
)

## Interpretation and intended follow-up

1. `D_REF_NM` is calculated from the 74.0° experimental state. Changing
   `REFERENCE["gamma_deg"]`, `REFERENCE["a_nm"]`, the roundness or the particle
   size automatically recalibrates the reference distance.
2. `nearest_exact_gap` is the true Euclidean distance for two identically
   oriented Minkowski rounded cubes. `nearest_support_gap` is retained only to
   show why the old 1.8 nm support-plane constraint suppressed intermediate
   rhombohedral angles.
3. `gamma=0` is the true simple-cubic face-aligned state at 90°. The reference
   V6.02 frame is `gamma=-15°`.
4. The default `N_coherent_particles=1` free-energy curve is a local-rotor
   approximation. Increasing this parameter makes the per-particle free energy
   approach the coherent `min_gamma E` limit; it should not be interpreted as
   an extensive bulk rotational entropy without choosing a domain size.
5. The sharp \(4^3\) voxel VdW model is intentionally retained. If quantitative
   fitting requires it, the next controlled change is a rounded occupancy mask
   at the same voxel resolution—not an immediate increase in `N_voxel`.
6. No contact-area-dependent ligand repulsion is fitted here. If the exact-gap
   model still over-stabilizes the six SC face contacts, add that term in a
   separate sensitivity branch rather than absorbing it into \(M_s\) or the
   Hamaker constant.
7. The two detailed V6.03-style multi-panel outputs are repeated for
   \(L=13,\ 16,\ 19.5\) nm.
   All three sizes use the single \(L=16\) calibration
   \(D_{\rm exact}=D_{74}\); this assumes the same effective ligand-shell
   separation. The \(L=19.5\) nm experimental point at
   \((a,\alpha)=(29\,{\rm nm},60^\circ)\) is plotted only as validation.
8. The landscape uses the complete Hamiltonian
   \(E_Z+E_A+E_{dd}^{26}+E_{\rm VdW}^{6}\). In every component panel,
   however, the displayed black `Total` curve is defined strictly as
   \(E_{dd}^{26}+E_{\rm VdW}^{6}\), the sum of the two displayed curves.
   Dipole interactions use all 26 lattice neighbours, while VdW uses the six
   fixed primitive bonds \(\pm\mathbf a_1,\pm\mathbf a_2,\pm\mathbf a_3\).
   The exported full-26 VdW sum and tail-20 remainder are diagnostics only.
9. With this face-bond Hamiltonian, \(E_{\rm VdW}^{6}\) decreases
   monotonically from \(80^\circ\) to \(90^\circ\), and its minimum is at the
   face-aligned simple-cubic state \((\alpha,\gamma)=(90^\circ,0^\circ)\) for
   all three particle sizes. The old 80--90 degree plateau belongs to the
   full-26 diagnostic: the attractive tail from the other 20 neighbours
   offsets part of the six-face trend.
10. The hard constraint still checks the exact rounded-cube gap against all
    26 neighbours. Therefore \(D_{\min}=D_{74}\) stays fixed along the scan:
    from \(80^\circ\) to \(90^\circ\), it is the lattice parameter and face
    alignment/overlap that improve, not a further decrease of the minimum
    surface gap. Letting the gap itself relax requires replacing the hard
    boundary with a soft steric potential and optimizing \(a\).
11. For \(L=19.5\) nm and \(\alpha=60^\circ\), the common-gap calculation
    gives \(a\approx29.0\) nm, essentially the measured value. At
    \(60^\circ\), six non-primitive difference vectors are center-distance
    degenerate with the primitive bonds; the six-bond model is therefore an
    explicit primitive-bond approximation, not a dynamic six-nearest-neighbour
    selection.